# Train selector models using only 10 input features

This notebook trains and saves models using **only these pre-run input features**:

`num_nodes, num_edges, graph_density, avg_degree, num_connected_component, bandwidth, latency, failure_rate, loss_rate, hamiltonian_cycle_found`

It does **not** use `*_elections` or `*_failed_attempts` as input features.

It trains 3 tasks:

1. `time` — choose algorithm based on election time.
2. `msg` — choose algorithm based on message count.
3. `score` — choose algorithm based on combined time + message + failed-attempt penalty.

For each task it saves:

- `linear_regression_<task>.pkl`
- `logistic_regression_<task>.pkl`
- `dl_<task>.h5`
- sidecar metadata files for DL features/classes

In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

DATASET_PATH = "all_algo_w_input_gen/final_results_v5.csv"   # change if needed
MODELS_DIR = "models_10_features"
os.makedirs(MODELS_DIR, exist_ok=True)

ALGORITHMS = ["bully", "ring", "bidir_ring", "raft", "echo"]

INPUT_FEATURES = [
    "num_nodes",
    "num_edges",
    "graph_density",
    "avg_degree",
    "num_connected_component",
    "bandwidth",
    "latency",
    "failure_rate",
    "loss_rate",
    "hamiltonian_cycle_found",
]

TIME_COLS = {
    "bully": "bully_time",
    "ring": "ring_time",
    "bidir_ring": "bidir_ring_time",
    "raft": "raft_time",
    "echo": "echo_time",
}

MSG_COLS = {
    "bully": "bully_msg_count",
    "ring": "ring_msg_count",
    "bidir_ring": "bidir_ring_msg_count",
    "raft": "raft_msg_count",
    "echo": "echo_msg_count",
}

ELECTION_COLS = {
    "bully": "bully_elections",
    "ring": "ring_elections",
    "bidir_ring": "bidir_ring_elections",
    "raft": "raft_elections",
    "echo": "echo_elections",
}

FAILED_COLS = {
    "bully": "bully_failed_attempts",
    "ring": "ring_failed_attempts",
    "bidir_ring": "bidir_ring_failed_attempts",
    "raft": "raft_failed_attempts",
    "echo": "echo_failed_attempts",
}

TIME_WEIGHT = 1.0
MSG_WEIGHT = 1.0
FAILED_WEIGHT = 2.0
INVALID_SCORE = 10.0


In [5]:
df = pd.read_csv(DATASET_PATH)

missing = [c for c in INPUT_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing input feature columns: {missing}")

for col_map in [TIME_COLS, MSG_COLS, ELECTION_COLS, FAILED_COLS]:
    missing = [c for c in col_map.values() if c not in df.columns]
    if missing:
        raise ValueError(f"Missing output columns: {missing}")

df = df.copy()
df[INPUT_FEATURES] = df[INPUT_FEATURES].fillna(0)

print("Dataset shape:", df.shape)
print("Input features:", INPUT_FEATURES)
df.head()


Dataset shape: (9840, 30)
Input features: ['num_nodes', 'num_edges', 'graph_density', 'avg_degree', 'num_connected_component', 'bandwidth', 'latency', 'failure_rate', 'loss_rate', 'hamiltonian_cycle_found']


,num_nodes,num_edges,graph_density,avg_degree,num_connected_component,bandwidth,latency,failure_rate,loss_rate,hamiltonian_cycle_found,...,bidir_ring_elections,bidir_ring_failed_attempts,raft_time,raft_msg_count,raft_elections,raft_failed_attempts,echo_time,echo_msg_count,echo_elections,echo_failed_attempts
0,3,2,0.666667,1.333333,1,1.0,5.0,0.001,0.00,0,...,0,0,0.617832,6.000000,1,0,0.286278,6.0,1,0
1,3,2,0.666667,1.333333,1,1.0,5.0,0.001,0.03,0,...,0,0,0.576438,6.000000,1,0,0.261559,6.0,1,0
2,3,2,0.666667,1.333333,1,1.0,5.0,0.005,0.00,0,...,0,0,0.736494,4.500000,4,27,0.257219,2.4,5,18
3,3,2,0.666667,1.333333,1,1.0,5.0,0.005,0.03,0,...,0,0,0.663553,4.666667,3,18,0.353658,2.4,5,2
4,3,2,0.666667,1.333333,1,1.0,5.0,0.010,0.00,0,...,0,0,0.592130,4.500000,4,18,0.497381,2.4,5,14


In [6]:
def is_valid_algo_result(row, algo, metric_cols):
    metric = row[metric_cols[algo]]
    elections = row[ELECTION_COLS[algo]]

    if pd.isna(metric) or metric <= 0:
        return False

    if pd.isna(elections) or elections <= 0:
        return False

    if algo in ["ring", "bidir_ring"] and int(row["hamiltonian_cycle_found"]) != 1:
        return False

    return True


def best_algo_by_metric(row, metric_cols):
    best_algo = None
    best_value = np.inf

    for algo in ALGORITHMS:
        if is_valid_algo_result(row, algo, metric_cols):
            value = float(row[metric_cols[algo]])
            if value < best_value:
                best_value = value
                best_algo = algo

    return best_algo


def row_minmax_normalize(values):
    arr = np.array(values, dtype=float)
    finite = np.isfinite(arr)

    if finite.sum() == 0:
        return np.full_like(arr, INVALID_SCORE, dtype=float)

    mn = arr[finite].min()
    mx = arr[finite].max()

    out = np.full_like(arr, INVALID_SCORE, dtype=float)

    if mx == mn:
        out[finite] = 0.0
    else:
        out[finite] = (arr[finite] - mn) / (mx - mn)

    return out


def compute_score_targets(df):
    score_matrix = []

    for _, row in df.iterrows():
        raw_time = []
        raw_msg = []
        raw_fail = []

        for algo in ALGORITHMS:
            valid_time = is_valid_algo_result(row, algo, TIME_COLS)
            valid_msg = is_valid_algo_result(row, algo, MSG_COLS)
            valid = valid_time and valid_msg

            if valid:
                raw_time.append(float(row[TIME_COLS[algo]]))
                raw_msg.append(float(row[MSG_COLS[algo]]))
                completed = max(float(row[ELECTION_COLS[algo]]), 1.0)
                failed = max(float(row[FAILED_COLS[algo]]), 0.0)
                raw_fail.append(failed / (completed + failed + 1e-9))
            else:
                raw_time.append(np.inf)
                raw_msg.append(np.inf)
                raw_fail.append(np.inf)

        time_norm = row_minmax_normalize(raw_time)
        msg_norm = row_minmax_normalize(raw_msg)
        fail_norm = row_minmax_normalize(raw_fail)

        score = (
            TIME_WEIGHT * time_norm
            + MSG_WEIGHT * msg_norm
            + FAILED_WEIGHT * fail_norm
        )

        score = np.where(np.isfinite(score), score, INVALID_SCORE)
        score = np.clip(score, 0, INVALID_SCORE)

        score_matrix.append(score)

    score_df = pd.DataFrame(score_matrix, columns=[f"{a}_score" for a in ALGORITHMS])
    return score_df


def best_algo_by_score_row(row):
    scores = {algo: row[f"{algo}_score"] for algo in ALGORITHMS}
    return min(scores, key=scores.get)


df["target_time"] = df.apply(lambda r: best_algo_by_metric(r, TIME_COLS), axis=1)
df["target_msg"] = df.apply(lambda r: best_algo_by_metric(r, MSG_COLS), axis=1)

score_df = compute_score_targets(df)
df = pd.concat([df, score_df], axis=1)
df["target_score"] = df.apply(best_algo_by_score_row, axis=1)

print("Target distribution: TIME")
print(df["target_time"].value_counts(dropna=False))

print("\nTarget distribution: MSG")
print(df["target_msg"].value_counts(dropna=False))

print("\nTarget distribution: SCORE")
print(df["target_score"].value_counts(dropna=False))


Target distribution: TIME
target_time
raft          5106
bidir_ring    2261
echo          1492
bully          523
NaN            249
ring           209
Name: count, dtype: int64

Target distribution: MSG
target_msg
ring          4499
raft          3689
bidir_ring     683
echo           486
NaN            249
bully          234
Name: count, dtype: int64

Target distribution: SCORE
target_score
raft          7143
bully         1218
echo          1025
bidir_ring     380
ring            74
Name: count, dtype: int64


In [7]:
def make_regression_target_matrix(df, metric_cols, task):
    Y = pd.DataFrame(index=df.index)

    valid_values = []
    for algo in ALGORITHMS:
        col = metric_cols[algo]
        vals = df[col]
        valid_values.extend(vals[(vals > 0) & vals.notna()].tolist())

    if len(valid_values) == 0:
        raise ValueError(f"No valid values found for task={task}")

    invalid_value = float(np.percentile(valid_values, 95) * 3.0)
    invalid_value = max(invalid_value, 1.0)

    for algo in ALGORITHMS:
        col = metric_cols[algo]
        y = []

        for _, row in df.iterrows():
            if is_valid_algo_result(row, algo, metric_cols):
                y.append(float(row[col]))
            else:
                y.append(invalid_value)

        Y[algo] = y

    return Y, invalid_value


Y_time, INVALID_TIME_VALUE = make_regression_target_matrix(df, TIME_COLS, "time")
Y_msg, INVALID_MSG_VALUE = make_regression_target_matrix(df, MSG_COLS, "msg")
Y_score = df[[f"{a}_score" for a in ALGORITHMS]].copy()
Y_score.columns = ALGORITHMS

print("Invalid time fill value:", INVALID_TIME_VALUE)
print("Invalid msg fill value:", INVALID_MSG_VALUE)

display(Y_time.head())
display(Y_msg.head())
display(Y_score.head())


Invalid time fill value: 27.7114947
Invalid msg fill value: 3551.903999549997


,bully,ring,bidir_ring,raft,echo
0,0.289451,27.711495,27.711495,0.617832,0.286278
1,0.289367,27.711495,27.711495,0.576438,0.261559
2,0.260758,27.711495,27.711495,0.736494,0.257219
3,0.257420,27.711495,27.711495,0.663553,0.353658
4,0.459798,27.711495,27.711495,0.592130,0.497381


,bully,ring,bidir_ring,raft,echo
0,8.000000,3551.904,3551.904,6.000000,6.0
1,8.000000,3551.904,3551.904,6.000000,6.0
2,2.833333,3551.904,3551.904,4.500000,2.4
3,2.800000,3551.904,3551.904,4.666667,2.4
4,2.833333,3551.904,3551.904,4.500000,2.4


,bully,ring,bidir_ring,raft,echo
0,1.009570,10.0,10.0,1.0,0.000000
1,1.088313,10.0,10.0,1.0,0.000000
2,0.213733,10.0,10.0,4.0,1.347826
3,0.176471,10.0,10.0,4.0,0.903628
4,0.206349,10.0,10.0,4.0,1.836637


In [8]:
def choose_from_pred_matrix(pred_matrix):
    indices = np.argmin(pred_matrix, axis=1)
    return np.array([ALGORITHMS[i] for i in indices])


def train_linear_regression_set(task_name, Y):
    X = df[INPUT_FEATURES].astype(float)

    valid_rows = Y.notna().all(axis=1)
    X = X[valid_rows]
    Y_task = Y[valid_rows]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X,
        Y_task,
        test_size=0.2,
        random_state=42,
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    models_by_algo = {}
    pred_test = []

    for algo in ALGORITHMS:
        model = LinearRegression()
        model.fit(X_train_s, Y_train[algo].values)
        models_by_algo[algo] = model
        pred_test.append(model.predict(X_test_s))

    pred_matrix = np.vstack(pred_test).T
    y_pred_algo = choose_from_pred_matrix(pred_matrix)
    y_true_algo = choose_from_pred_matrix(Y_test[ALGORITHMS].values)

    acc = accuracy_score(y_true_algo, y_pred_algo)

    bundle = {
        "task": task_name,
        "model_type": "linear_regression_set",
        "features": INPUT_FEATURES,
        "scaler": scaler,
        "models_by_algo": models_by_algo,
        "algos": ALGORITHMS,
        "note": "Uses only 10 pre-run input features. Predicts per-algorithm metric and selects argmin.",
    }

    out_path = os.path.join(MODELS_DIR, f"linear_regression_{task_name}.pkl")
    joblib.dump(bundle, out_path)

    print(f"\nSaved: {out_path}")
    print(f"Linear regression {task_name} argmin accuracy:", acc)

    return acc


def train_logistic_classifier(task_name, target_col):
    data = df[df[target_col].notna()].copy()

    X = data[INPUT_FEATURES].astype(float)
    y_label = data[target_col].astype(str).values

    X_train, X_test, y_train_label, y_test_label = train_test_split(
        X,
        y_label,
        test_size=0.2,
        random_state=42,
        stratify=y_label,
    )

    le = LabelEncoder()
    y_train = le.fit_transform(y_train_label)
    y_test = le.transform(y_test_label)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = LogisticRegression(
        solver="lbfgs",
        max_iter=3000,
        random_state=42,
    )

    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    pred_label = le.inverse_transform(pred)

    acc = accuracy_score(y_test_label, pred_label)

    bundle = {
        "task": task_name,
        "model_type": "logistic_regression_classifier",
        "features": INPUT_FEATURES,
        "scaler": scaler,
        "model": model,
        "classes": list(le.classes_),
        "label_encoder": le,
        "note": "Uses only 10 pre-run input features.",
    }

    out_path = os.path.join(MODELS_DIR, f"logistic_regression_{task_name}.pkl")
    joblib.dump(bundle, out_path)

    print(f"\nSaved: {out_path}")
    print(f"Logistic regression {task_name} accuracy:", acc)
    print(classification_report(y_test_label, pred_label))

    return acc


def train_dl_classifier(task_name, target_col, epochs=60):
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, Sequential
    except Exception as e:
        print(f"\nSkipping DL {task_name}: TensorFlow is not installed or not working.")
        print("Reason:", e)
        return None

    data = df[df[target_col].notna()].copy()

    X = data[INPUT_FEATURES].astype(float).values
    y_label = data[target_col].astype(str).values

    le = LabelEncoder()
    y = le.fit_transform(y_label)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    normalizer = layers.Normalization()
    normalizer.adapt(X_train)

    model = Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        normalizer,
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.25),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.25),
        layers.Dense(16, activation="relu"),
        layers.Dense(len(le.classes_), activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        verbose=0,
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)

    h5_path = os.path.join(MODELS_DIR, f"dl_{task_name}.h5")
    model.save(h5_path)

    with open(os.path.join(MODELS_DIR, f"dl_{task_name}_features.txt"), "w") as f:
        for c in INPUT_FEATURES:
            f.write(c + "\n")

    with open(os.path.join(MODELS_DIR, f"dl_{task_name}_classes.txt"), "w") as f:
        for c in le.classes_:
            f.write(str(c) + "\n")

    joblib.dump(
        {
            "task": task_name,
            "features": INPUT_FEATURES,
            "classes": list(le.classes_),
            "label_encoder": le,
        },
        os.path.join(MODELS_DIR, f"dl_{task_name}_metadata.pkl")
    )

    print(f"\nSaved: {h5_path}")
    print(f"DL {task_name} accuracy:", acc)

    return acc


In [9]:
results = {}

results["linear_regression_time"] = train_linear_regression_set("time", Y_time)
results["logistic_regression_time"] = train_logistic_classifier("time", "target_time")
results["dl_time"] = train_dl_classifier("time", "target_time", epochs=60)

results["linear_regression_msg"] = train_linear_regression_set("msg", Y_msg)
results["logistic_regression_msg"] = train_logistic_classifier("msg", "target_msg")
results["dl_msg"] = train_dl_classifier("msg", "target_msg", epochs=60)

results["linear_regression_score"] = train_linear_regression_set("score", Y_score)
results["logistic_regression_score"] = train_logistic_classifier("score", "target_score")
results["dl_score"] = train_dl_classifier("score", "target_score", epochs=60)

print("\n======================")
print("Training summary")
print("======================")
for k, v in results.items():
    print(f"{k}: {v}")

pd.DataFrame.from_dict(results, orient="index", columns=["accuracy"])



Saved: models_10_features\linear_regression_time.pkl
Linear regression time argmin accuracy: 0.5757113821138211

Saved: models_10_features\logistic_regression_time.pkl
Logistic regression time accuracy: 0.7050547159979156
              precision    recall  f1-score   support

  bidir_ring       0.60      0.62      0.61       452
       bully       0.40      0.18      0.25       105
        echo       0.65      0.62      0.64       298
        raft       0.78      0.85      0.81      1022
        ring       0.00      0.00      0.00        42

    accuracy                           0.71      1919
   macro avg       0.48      0.45      0.46      1919
weighted avg       0.68      0.71      0.69      1919




Saved: models_10_features\dl_time.h5
DL time accuracy: 0.7529963254928589

Saved: models_10_features\linear_regression_msg.pkl
Linear regression msg argmin accuracy: 0.3551829268292683

Saved: models_10_features\logistic_regression_msg.pkl
Logistic regression msg accuracy: 0.7269411151641479
              precision    recall  f1-score   support

  bidir_ring       0.00      0.00      0.00       137
       bully       0.62      0.17      0.27        47
        echo       0.65      0.25      0.36        97
        raft       0.71      0.74      0.72       738
        ring       0.74      0.91      0.82       900

    accuracy                           0.73      1919
   macro avg       0.54      0.41      0.43      1919
weighted avg       0.67      0.73      0.69      1919




Saved: models_10_features\dl_msg.h5
DL msg accuracy: 0.7514330148696899

Saved: models_10_features\linear_regression_score.pkl
Linear regression score argmin accuracy: 0.7494918699186992

Saved: models_10_features\logistic_regression_score.pkl
Logistic regression score accuracy: 0.7723577235772358
              precision    recall  f1-score   support

  bidir_ring       0.38      0.04      0.07        76
       bully       0.58      0.29      0.38       244
        echo       0.56      0.28      0.38       205
        raft       0.80      0.97      0.88      1428
        ring       0.00      0.00      0.00        15

    accuracy                           0.77      1968
   macro avg       0.46      0.32      0.34      1968
weighted avg       0.73      0.77      0.73      1968




Saved: models_10_features\dl_score.h5
DL score accuracy: 0.7911585569381714

Training summary
linear_regression_time: 0.5757113821138211
logistic_regression_time: 0.7050547159979156
dl_time: 0.7529963254928589
linear_regression_msg: 0.3551829268292683
logistic_regression_msg: 0.7269411151641479
dl_msg: 0.7514330148696899
linear_regression_score: 0.7494918699186992
logistic_regression_score: 0.7723577235772358
dl_score: 0.7911585569381714


,accuracy
linear_regression_time,0.575711
logistic_regression_time,0.705055
dl_time,0.752996
linear_regression_msg,0.355183
logistic_regression_msg,0.726941
dl_msg,0.751433
linear_regression_score,0.749492
logistic_regression_score,0.772358
dl_score,0.791159


In [10]:
print("Saved files in:", MODELS_DIR)
for f in sorted(os.listdir(MODELS_DIR)):
    print(f)


Saved files in: models_10_features
dl_msg.h5
dl_msg_classes.txt
dl_msg_features.txt
dl_msg_metadata.pkl
dl_score.h5
dl_score_classes.txt
dl_score_features.txt
dl_score_metadata.pkl
dl_time.h5
dl_time_classes.txt
dl_time_features.txt
dl_time_metadata.pkl
linear_regression_msg.pkl
linear_regression_score.pkl
linear_regression_time.pkl
logistic_regression_msg.pkl
logistic_regression_score.pkl
logistic_regression_time.pkl
